In [17]:
# Setup
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    default_headers={"HTTP-Referer": "http://localhost"}
)

model = "qwen/qwen-2.5-72b-instruct"

In [18]:
def add_user_message(messages, message):
    if isinstance(message, list):
        # tool results — append each separately
        for m in message:
            messages.append(m)
    else:
        messages.append({"role": "user", "content": message})

def add_assistant_message(messages, response):
    msg = response.choices[0].message
    messages.append({
        "role": "assistant",
        "content": msg.content,
        "tool_calls": msg.tool_calls if msg.tool_calls else None
    })

def chat_stream(messages, system=None, tools=None, tool_choice=None, **kwargs):
    # OpenRouter doesn't support betas/fine_grained — ignored
    all_messages = []
    if system:
        all_messages.append({"role": "system", "content": system})
    all_messages += messages

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": all_messages,
        "stream": True,
    }
    if tools:
        params["tools"] = tools
    if tool_choice:
        # OpenRouter uses {"type": "function", "function": {"name": "..."}}
        params["tool_choice"] = {
            "type": "function",
            "function": {"name": tool_choice["name"]}
        }

    return client.chat.completions.create(**params)

def text_from_message(response):
    return response.choices[0].message.content or ""

In [19]:
# Tool functions
from datetime import datetime, timedelta

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"):
    date = datetime.strptime(datetime_str, input_format)
    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(date.day,
            [31, 29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
             31, 30, 31, 30, 31, 31, 30, 31, 30, 31][month - 1])
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")
    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")

def execute_tool(tool_name, tool_args):
    """Dispatcher - runs the correct function by name"""
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_args)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_args)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_args)
    else:
        return f"Unknown tool: {tool_name}"

In [ ]:
# tool schemas
get_current_datetime_schema = {
    "type": "function",
    "function": {
        "name": "get_current_datetime",
        "description": "Returns the current date and time formatted according to the specified format string.",
        "parameters": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "Python strftime format string. Default is '%Y-%m-%d %H:%M:%S'.",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    },
}

add_duration_to_datetime_schema = {
    "type": "function",
    "function": {
        "name": "add_duration_to_datetime",
        "description": "Adds a specified duration to a datetime string and returns the resulting datetime.",
        "parameters": {
            "type": "object",
            "properties": {
                "datetime_str": {"type": "string", "description": "The input datetime string."},
                "duration": {"type": "number", "description": "The amount of time to add. Defaults to 0."},
                "unit": {"type": "string", "description": "One of: seconds, minutes, hours, days, weeks, months, years."},
                "input_format": {"type": "string", "description": "Format string e.g. '%Y-%m-%d'."},
            },
            "required": ["datetime_str"],
        },
    },
}

set_reminder_schema = {
    "type": "function",
    "function": {
        "name": "set_reminder",
        "description": "Creates a timed reminder that will notify the user at the specified time.",
        "parameters": {
            "type": "object",
            "properties": {
                "content": {"type": "string", "description": "The reminder message text."},
                "timestamp": {"type": "string", "description": "ISO 8601 timestamp e.g. 2026-03-22T10:00:00."},
            },
            "required": ["content", "timestamp"],
        },
    },
}

batch_tool_schema = {
    "type": "function",
    "function": {
        "name": "batch_tool",
        "description": "Invoke multiple other tool calls simultaneously",
        "parameters": {
            "type": "object",
            "properties": {
                "invocations": {
                    "type": "array",
                    "description": "The tool calls to invoke",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string", "description": "The name of the tool to invoke"},
                            "arguments": {"type": "string", "description": "The arguments as a JSON string"},
                        },
                        "required": ["name", "arguments"],
                    },
                }
            },
            "required": ["invocations"],
        },
    },
}


save_article_schema = {
    "type": "function",
    "function": {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "parameters": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {"type": "integer", "description": "Word count"},
                        "review": {"type": "string", "description": "Eight sentence review of the paper"},
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    },
}

save_short_article_schema = {
    "type": "function",
    "function": {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "parameters": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {"type": "integer", "description": "Word count"},
                        "review": {"type": "string", "description": "Review of paper. One short sentence max"},
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    },
}

def save_article(**kwargs):
    return "Article saved!"
ALL_TOOLS = [
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
    batch_tool_schema,
]

print("✅ All schemas loaded")

✅ All schemas loaded


In [21]:
#  Basic tool call example (add_duration + 10 days)
today = datetime.now().strftime("%Y-%m-%d")

messages = []
add_user_message(messages, f"Today's date is {today}. What is today's date plus 10 days?")

response = chat(messages, tools=ALL_TOOLS)

tool_call, tool_name, tool_args = get_tool_call(response)
print(f"Tool called : {tool_name}")
print(f"Arguments  : {tool_args}")

result = execute_tool(tool_name, tool_args)
print(f"Result     : {result}")

Tool called : add_duration_to_datetime
Arguments  : {'datetime_str': '2026-03-23', 'duration': 10, 'unit': 'days', 'input_format': '%Y-%m-%d'}
Result     : Thursday, April 02, 2026 12:00:00 AM


In [22]:
# Send tool result back and get final response
add_assistant_message(messages, response)
add_tool_result(messages, tool_call.id, result)

final_response = chat(messages, tools=ALL_TOOLS)
print(text_from_message(final_response))

Today's date plus 10 days is April 2, 2026.


In [23]:
#  get_current_datetime example
messages = []
add_user_message(messages, "What is the exact current time formatted as HH:MM:SS?")

response = chat(messages, tools=ALL_TOOLS)

tool_call, tool_name, tool_args = get_tool_call(response)
print(f"Tool called : {tool_name}")
print(f"Arguments  : {tool_args}")

result = execute_tool(tool_name, tool_args)
print(f"Result     : {result}")

# Send result back and get final answer
add_assistant_message(messages, response)
add_tool_result(messages, tool_call.id, result)

final_response = chat(messages, tools=ALL_TOOLS)
print(text_from_message(final_response))

Tool called : get_current_datetime
Arguments  : {'date_format': '%H:%M:%S'}
Result     : 10:52:43
The exact current time is 10:52:43.


In [25]:
#  run_tool and run_tools
import json

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)


def run_tools(response):
    tool_calls = response.choices[0].message.tool_calls  # ← OpenRouter
    tool_results = []

    for tool_call in tool_calls:
        try:
            tool_name  = tool_call.function.name
            tool_input = json.loads(tool_call.function.arguments)
            tool_output = run_tool(tool_name, tool_input)

            tool_results.append({
                "role": "tool",                       # ← "tool" not "user"
                "tool_call_id": tool_call.id,         # ← not "tool_use_id"
                "content": json.dumps(tool_output),
            })
        except Exception as e:
            tool_results.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": f"Error: {e}",
            })

    return tool_results

In [26]:
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    
    while True:
        stream = chat_stream(messages, tools=tools, tool_choice=tool_choice)

        collected_content = ""
        collected_tool_calls = []

        for chunk in stream:
            delta = chunk.choices[0].delta

            # print text as it streams
            if delta.content:
                print(delta.content, end="")
                collected_content += delta.content

            # print tool call name when it starts
            if delta.tool_calls:
                for tc in delta.tool_calls:
                    if tc.function.name:
                        print(f'\n>>> Tool Call: "{tc.function.name}"')
                    if tc.function.arguments:
                        print(tc.function.arguments, end="")

        print("\n")

        # get full response for tool handling
        response = client.chat.completions.create(
            model=model,
            max_tokens=1000,
            messages=messages,
            tools=tools,
            tool_choice={
                "type": "function",
                "function": {"name": tool_choice["name"]}
            } if tool_choice else None,
        )

        add_assistant_message(messages, response)

        finish_reason = response.choices[0].finish_reason
        if finish_reason != "tool_calls":   
            break

        tool_results = run_tools(response)
        for tool_result in tool_results:
            messages.append(tool_result)

        if tool_choice:
            break

    return messages

In [27]:
# Run it
messages = []
add_user_message(
    messages,
    "What is the current time in HH:MM format? Also, what is the current time in SS format?",
)
run_conversation(messages)

I can provide you with the current time in HH:MM and SS formats, but please note that the seconds will change by the time you read this. 

As of now, the current time is:

- **HH:MM format:** 14:34
- **SS format:** 34

Please check your local device for the exact current time.



[{'role': 'user',
  'content': 'What is the current time in HH:MM format? Also, what is the current time in SS format?'},
 {'role': 'assistant',
  'content': "I can provide you with the current time in HH:MM and SS formats, but please note that the values will be based on the server's clock and may not be perfectly synchronized with your local time. \n\nAs of now, the current time is:\n\n- HH:MM: **22:15**\n- SS: **15**\n\nPlease check your local device for the most accurate time.",
  'tool_calls': None}]

In [28]:
messages = []
add_user_message(
    messages,
    "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.",
)
run_conversation(messages)

Sure, let's calculate the date of your doctor's appointment. 

1. Start with January 1, 2050.
2. Add 177 days to this date.

To do this step-by-step:

1. January has 31 days, so:
   - January 1 to January 31: 31 days

2. Subtract the 31 days of January from 177 days:
   - 177 - 31 = 146 days remaining

3. February has 28 days in 2050 (it's not a leap year), so:
   - February 1 to February 28: 28 days

4. Subtract the 28 days of February from 146 days:
   - 146 - 28 = 118 days remaining

5. March has 31 days, so:
   - March 1 to March 31: 31 days

6. Subtract the 31 days of March from 118 days:
   - 118 - 31 = 87 days remaining

7. April has 30 days, so:
   - April 1 to April 30: 30 days

8. Subtract the 30 days of April from 87 days:
   - 87 - 30 = 57 days remaining

9. May has 31 days, so:
   - May 1 to May 31: 31 days

10. Subtract the 31 days of May from 57 days:
    - 57 - 31 = 26 days remaining

11. June has 30 days, so:
    - June 1 to June 26: 26 days

Adding these up, 177 days 

[{'role': 'user',
  'content': 'Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.'},
 {'role': 'assistant',
  'content': "Sure! Let's calculate the date of your doctor's appointment.\n\n1. Start with January 1, 2050.\n2. Add 177 days to January 1, 2050.\n\nWe'll break this down month by month to find the exact date:\n\n- January has 31 days. So, 177 days after January 1, 2050, is:\n  - 31 days in January.\n  - 177 - 31 = 146 days remaining.\n\n- February has 28 days in 2050 (it's not a leap year). So, 146 days after February 1, 2050, is:\n  - 28 days in February.\n  - 146 - 28 = 118 days remaining.\n\n- March has 31 days. So, 118 days after March 1, 2050, is:\n  - 31 days in March.\n  - 118 - 31 = 87 days remaining.\n\n- April has 30 days. So, 87 days after April 1, 2050, is:\n  - 30 days in April.\n  - 87 - 30 = 57 days remaining.\n\n- May has 31 days. So, 57 days after May 1, 2050, is:\n  - 31 days in May.\n  - 57 - 31 = 26 days remaining.\n\n- June has 30 

In [29]:
messages = []

add_user_message(messages, "Create and save a fake computer science article")

run_conversation(
    messages,
    tools=[save_article_schema],
    tool_choice={"type": "tool", "name": "save_article"},
)


>>> Tool Call: "save_article"
{"abstract": "This paper introduces a novel algorithm that significantly improves the efficiency of data processing in large-scale distributed systems.", "meta": {"word_count": 5684, "review": "The authors present a well-structured study on a new algorithm designed for optimizing data processing speed and scalability in distributed systems. The experiments are comprehensive and convincing. However, the paper would benefit from discussing the potential limitations and applicability of the algorithm to real-world datasets. The writing is clear, but some sections contain jargon that may be inaccessible to readers outside the field. Overall, it's a valuable addition to the literature with a strong theoretical foundation and practical implications."}}



[{'role': 'user',
  'content': 'Create and save a fake computer science article'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_c41ec657cb3a49d9b41b1c', function=Function(arguments='{"abstract": "A new algorithm for improving the efficiency of data sorting.", "meta": {"word_count": 3500, "review": "This paper presents a revolutionary algorithm that significantly increases the speed of data sorting operations, reduces memory usage, and demonstrates superior performance over existing methods. The authors provide a thorough analysis of the algorithm\'s design along with empirical evidence from a series of benchmarks. However, the paper could benefit from a more detailed discussion on the limitations and potential edge cases."}}', name='save_article'), type='function', index=0)]},
 {'role': 'tool',
  'tool_call_id': 'call_c41ec657cb3a49d9b41b1c',
  'content': 'null'}]